In [1]:
# === Supervised Contrastive projection + K-shot evaluation (participant-wise CV) ===

In [2]:
!nvidia-smi

Sat Nov  1 20:51:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import numpy as np
import pandas as pd
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.utils import to_categorical

In [8]:
# ---- 1) Load participants & embeddings ----
participants_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Autism/participants.csv')
X_participants = participants_df["ParticipantID"].astype(str).values
y_participants = participants_df["Label"].astype(int).values  # 0/1 label per participant

ds = np.load('/content/drive/MyDrive/Colab Notebooks/Autism/Few-Shot/ASD_vgg16.npz', allow_pickle=True)
X   = ds['X'].astype('float32')   # embeddings
y   = ds['y'].astype('int32')     # 0=TD, 1=ASD
IDs = ds['IDs'].astype(str)       # per-image participant ID

In [9]:
# L2-normalize embeddings for cosine similarity
X_base = normalize(X, norm='l2', axis=1).astype('float32')

print("Embeddings:", X_base.shape, "| images:", len(y), "| participants:", len(np.unique(IDs)))

# ---- 2) CV setup (participant-wise) ----
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

Embeddings: (3282, 512) | images: 3282 | participants: 54


In [10]:
K = tf.keras.backend

In [11]:
# ------------------ SupCon custom layer ------------------
class SupConLossLayer(layers.Layer):
    def __init__(self, temperature=0.2, **kwargs):
        super().__init__(**kwargs)
        self.temperature = temperature

    def call(self, inputs):
        z, y_onehot = inputs  # z: [B,d] L2-normalized; y_onehot: [B,2]
        eps = 1e-8
        sim = tf.matmul(z, z, transpose_b=True) / self.temperature  # cosine sim since z is normalized

        bsz = tf.shape(z)[0]
        eye = tf.eye(bsz)
        not_self = 1.0 - eye
        pos_mask = tf.matmul(y_onehot, y_onehot, transpose_b=True) * not_self  # same-class, not self

        exp_sim = tf.exp(sim) * not_self
        pos_sum = tf.reduce_sum(exp_sim * pos_mask, axis=1, keepdims=True) + eps
        denom   = tf.reduce_sum(exp_sim, axis=1, keepdims=True) + eps
        pos_cnt = tf.maximum(tf.reduce_sum(pos_mask, axis=1, keepdims=True), eps)

        loss_i = - tf.math.log(pos_sum / denom) / pos_cnt
        self.add_loss(tf.reduce_mean(loss_i))
        return z  # pass through embeddings

In [12]:
def build_supcon_head(feat_dim=512, emb_dim=128, temperature=0.2, dropout=0.2):
    feat_in = Input(shape=(feat_dim,), name='feat_in')
    y_in    = Input(shape=(2,), name='y_in')  # one-hot labels (for loss only)

    z = layers.Dense(256, activation='relu')(feat_in)
    if dropout and dropout > 0:
        z = layers.Dropout(dropout)(z)
    z = layers.Dense(emb_dim, use_bias=False)(z)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1), name='z_norm')(z)

    z_out = SupConLossLayer(temperature=temperature, name='supcon_loss')([z, y_in])
    model = Model([feat_in, y_in], z_out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=None)

    # Inference model: project features -> normalized embedding
    embed = Model(feat_in, z)
    return model, embed

In [13]:
# ------------------ Balanced tf.data dataset (TRAIN only) ------------------
def make_balanced_dataset(X_train, y_train, batch_size=64):
    feat_dim = X_train.shape[1]
    cls0 = np.where(y_train == 0)[0]
    cls1 = np.where(y_train == 1)[0]
    half = batch_size // 2
    rep0 = (half > len(cls0))
    rep1 = (half > len(cls1))

    def gen():
        while True:
            i0 = np.random.choice(cls0, size=half, replace=rep0)
            i1 = np.random.choice(cls1, size=half, replace=rep1)
            idx = np.concatenate([i0, i1])
            np.random.shuffle(idx)
            xb = X_train[idx].astype('float32')
            yb = to_categorical(y_train[idx], num_classes=2).astype('float32')
            yield xb, yb  # two tensors matching the model's two inputs

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(None, feat_dim), dtype=tf.float32),  # xb
            tf.TensorSpec(shape=(None, 2),        dtype=tf.float32),  # yb (one-hot)
        ),
    )
    # Keras expects a tuple whose first element is the "inputs" tuple
    ds = ds.map(lambda xb, yb: ((xb, yb),))
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

In [14]:
# ------------------ K-shot evaluation (cosine prototypes) ------------------
def kshot_eval(Z_train, y_train, Z_test, y_test, K_support=5, repeats=30, temperature=0.2):
    tr0 = np.where(y_train == 0)[0]
    tr1 = np.where(y_train == 1)[0]
    if len(tr0) == 0 or len(tr1) == 0 or len(Z_test) == 0:
        return None

    logits_accum = np.zeros((len(Z_test), 2), dtype='float32')
    for _ in range(repeats):
        s0 = np.random.choice(tr0, size=K_support, replace=(K_support > len(tr0)))
        s1 = np.random.choice(tr1, size=K_support, replace=(K_support > len(tr1)))

        P0 = Z_train[s0].mean(axis=0, keepdims=True); P0 /= (np.linalg.norm(P0, axis=1, keepdims=True) + 1e-8)
        P1 = Z_train[s1].mean(axis=0, keepdims=True); P1 /= (np.linalg.norm(P1, axis=1, keepdims=True) + 1e-8)

        logit0 = (Z_test @ P0.T) / temperature
        logit1 = (Z_test @ P1.T) / temperature
        logits_accum += np.hstack([logit0, logit1])

    logits = logits_accum / float(repeats)
    exps = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = exps[:, 1] / (exps.sum(axis=1) + 1e-8)

    auroc = roc_auc_score(y_test, probs) if len(np.unique(y_test)) > 1 else np.nan
    auprc = average_precision_score(y_test, probs)
    y_pred = (probs >= 0.5).astype(int)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    return auroc, auprc, prec, rec

In [19]:
# ------------------ CV + train SupCon per fold + K-shot eval ------------------
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
K_list = [1, 3, 5, 10]
repeats = 30
temperature = 0.2

In [20]:
# Training params (you can bump for more gains)
batch_size = 64
steps_per_epoch = 200
epochs = 20
emb_dim = 128


In [21]:
for Kshot in K_list:
    print(f"\n=== SupCon + {Kshot}-shot evaluation ===")
    aucs, auprcs, precs, recs = [], [], [], []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_participants, y_participants), 1):
        train_p = set(X_participants[tr_idx]); test_p = set(X_participants[te_idx])
        train_mask = np.array([pid in train_p for pid in IDs])
        test_mask  = np.array([pid in test_p for pid in IDs])

        X_tr, y_tr = X_base[train_mask], y[train_mask]
        X_te, y_te = X_base[test_mask],  y[test_mask]

        if X_tr.shape[0] == 0 or X_te.shape[0] == 0:
            print(f"Fold {fold}: no data — skipped.")
            continue
        if len(np.unique(y_tr)) < 2:
            print(f"Fold {fold}: train has single class — skipped.")
            continue

        # Build & train SupCon head on TRAIN images only
        K.clear_session()
        supcon_model, embed_model = build_supcon_head(
            feat_dim=X_tr.shape[1], emb_dim=emb_dim, temperature=temperature, dropout=0.2
        )
        ds = make_balanced_dataset(X_tr, y_tr, batch_size=batch_size)
        supcon_model.fit(ds, steps_per_epoch=steps_per_epoch, epochs=epochs, verbose=1)

        # Project to learned space (already L2-normalized by z_norm)
        Z_tr = embed_model.predict(X_tr, batch_size=512, verbose=0)
        Z_te = embed_model.predict(X_te, batch_size=512, verbose=0)

        res = kshot_eval(Z_tr, y_tr, Z_te, y_te, K_support=Kshot, repeats=repeats, temperature=temperature)
        if res is None:
            print(f"Fold {fold}: eval skipped (class missing or empty).")
            continue

        auroc, auprc, prec, rec = res
        aucs.append(auroc); auprcs.append(auprc); precs.append(prec); recs.append(rec)
        auroc_txt = f"{auroc:.3f}" if not np.isnan(auroc) else "n/a"
        print(f"Fold {fold}: AUROC={auroc_txt} | AUPRC={auprc:.3f} | Precision={prec:.3f} | Recall={rec:.3f} "
              f"[train n={len(y_tr)}, test n={len(y_te)}]")

    # Summary for this K
    def mean_sd(arr):
        arr = np.array(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        return (np.mean(arr), np.std(arr)) if len(arr) else (np.nan, np.nan)

    m_auc, s_auc = mean_sd(aucs)
    m_prc, s_prc = mean_sd(auprcs)
    m_p, s_p     = mean_sd(precs)
    m_r, s_r     = mean_sd(recs)
    auc_txt = f"{m_auc:.3f} ± {s_auc:.3f}" if not np.isnan(m_auc) else "n/a"
    print(f"== SupCon + {Kshot}-shot Summary ==  AUROC: {auc_txt} | AUPRC: {m_prc:.3f} ± {s_prc:.3f} | "
          f"Precision: {m_p:.3f} ± {s_p:.3f} | Recall: {m_r:.3f} ± {s_r:.3f}")



=== SupCon + 1-shot evaluation ===
Epoch 1/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0135
Epoch 2/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0015
Epoch 3/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 9.6924e-04
Epoch 4/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5.1441e-04
Epoch 5/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 6.7533e-04
Epoch 6/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 4.2210e-04
Epoch 7/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 5.0724e-04
Epoch 8/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 5.2017e-04
Epoch 9/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 3.4641e-04
Epoch 10/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.5391e-04
Epoch 11/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.4122e-04
Epoch 12/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.3384e-04
Epoch 13/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.8020e-04
Epoch 14/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2